# Pelajaran 11 - Protokol Agent-ke-Agent (A2A)


## Pengaturan


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## Apa itu Protokol A2A?

**Protokol Agen-ke-Agen (A2A)** adalah standar terbuka yang memungkinkan agen AI untuk berkomunikasi,
saling menemukan, dan berkolaborasi — bahkan ketika mereka dibangun di atas kerangka kerja yang berbeda atau dihosting
oleh layanan yang berbeda.

Konsep utama:

- **Penemuan** – Agen menerbitkan *Kartu Agen* yang menjelaskan kemampuan mereka, sehingga
  memudahkan agen lain (atau pengatur) untuk menemukan spesialis yang tepat untuk suatu tugas.
- **Pengiriman Pesan** – Agen bertukar pesan terstruktur melalui protokol umum, sehingga
  permintaan dari satu agen dapat dipahami dan dipenuhi oleh agen lain tanpa memandang
  implementasi internalnya.
- **Siklus Hidup Tugas** – A2A mendefinisikan status seperti *submitted* (diajukan), *working* (sedang dikerjakan), *completed* (selesai), dan
  *failed* (gagal), memberikan pengatur visibilitas penuh tentang bagaimana tugas yang didelegasikan sedang berjalan.

Dalam pelajaran ini kita mensimulasikan kolaborasi gaya A2A dengan menghubungkan tiga agen perjalanan khusus
ke dalam alur kerja di mana setiap agen memberikan keahliannya dan meneruskan hasil ke agen berikutnya.


## Membuat Agen Perjalanan Khusus


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Kolaborasi Multi-Agen melalui Alur Kerja

Kami menghubungkan ketiga agen ke dalam alur kerja berurutan yang mencerminkan pengiriman pesan A2A:

1. **CurrencyExchangeAgent** menerima permintaan pengguna dan menghasilkan panduan mata uang.
2. **ActivityPlannerAgent** menerima konteks yang diperkaya dan menambahkan rekomendasi aktivitas.
3. **TravelManagerAgent** menyintesis kedua input menjadi ringkasan perjalanan akhir.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Memahami A2A dalam Produksi

Dalam lingkungan produksi, protokol A2A membuka skenario lintas-layanan yang kuat:

| Kapabilitas | Deskripsi |
|---|---|
| **Interop antar-framework** | Agen yang dibangun dengan satu framework dapat mendelegasikan tugas ke agen yang dibangun dengan framework A2A-kompatibel lainnya, memungkinkan interoperabilitas lintas-organisasi yang sejati. |
| **Batas layanan** | Agen dapat berada di mikroservis terpisah, wilayah cloud, atau bahkan organisasi yang berbeda sekaligus tetap berkolaborasi secara mulus. |
| **Penemuan dinamis** | Sebuah pengatur dapat menanyakan registri Kartu Agen saat runtime untuk menemukan spesialis yang paling sesuai untuk sub-tugas tertentu. |
| **Streaming & notifikasi push** | A2A mendukung Server-Sent Events (SSE) untuk pembaruan progres waktu nyata dan notifikasi push untuk tugas yang berjalan lama. |

Alur kerja yang kami bangun di atas adalah versi sederhana, dalam-proses dari pola ini. Dalam
penyebaran nyata, setiap agen akan menampilkan endpoint HTTP, menerbitkan Kartu Agen, dan berkomunikasi
melalui protokol A2A JSON-RPC.


## Ringkasan

Dalam pelajaran ini Anda mempelajari:

1. **Apa itu protokol A2A** — standar terbuka untuk penemuan agen-ke-agen, pengiriman pesan,
   dan manajemen tugas.
2. **Cara membuat agen khusus** — agen Pertukaran Mata Uang, agen Perencana Kegiatan,
   dan pengatur Perjalanan.
3. **Cara menghubungkan agen ke dalam alur kerja** — menggunakan `WorkflowBuilder` untuk memodelkan
   pengiriman pesan berurutan antar agen.
4. **Cara kerja A2A dalam produksi** — memungkinkan kolaborasi lintas kerangka kerja,
   lintas layanan dengan penemuan dinamis dan pembaruan streaming.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan layanan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Meskipun kami berupaya untuk mencapai akurasi, harap diketahui bahwa terjemahan otomatis mungkin mengandung kesalahan atau ketidakakuratan. Dokumen asli dalam bahasa aslinya harus dianggap sebagai sumber yang sah. Untuk informasi penting, disarankan menggunakan terjemahan profesional oleh manusia. Kami tidak bertanggung jawab atas kesalahpahaman atau penafsiran yang keliru yang timbul dari penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
